# 🚗 Chatbot Konsultasi Kendaraan
Implementasi pipeline chatbot yang mengklasifikasi intent dan memberikan solusi berdasarkan keluhan user.

## 🔍 Load Intent Classifier (ONNX)

In [1]:
from transformers import AutoTokenizer
import numpy as np
import onnxruntime as ort
import torch

# Ganti ini sesuai model yang kamu pakai saat training
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")

# Load ONNX model
intent_session = ort.InferenceSession("../generated/intent_classifier.onnx")

# Intent labels harus sesuai urutan saat training
intent_labels = [
    "ask_first_aid_solution",
    "ask_possible_cause",
    "casual_greeting",
    "fallback",
    "goodbye",
    "report_noise_or_smell"
]


def predict_intent(text):
    # Tokenisasi dengan PyTorch, lalu convert ke numpy array
    inputs = tokenizer(
        text, 
        return_tensors="pt", 
        padding='max_length', 
        truncation=True, 
        max_length=128
    )
    inputs_onnx = {k: v.cpu().numpy() for k, v in inputs.items()}
    
    # Nama input ONNX
    input_ids_name = intent_session.get_inputs()[0].name
    attention_mask_name = intent_session.get_inputs()[1].name
    output_name = intent_session.get_outputs()[0].name

    # Jalankan inference
    outputs = intent_session.run([output_name], {
        input_ids_name: inputs_onnx["input_ids"],
        attention_mask_name: inputs_onnx["attention_mask"]
    })

    # Ambil prediksi kelas dengan skor tertinggi
    intent_idx = np.argmax(outputs[0], axis=1)[0]
    print("intent terdeteksi:", intent_labels[intent_idx])
    print("confidence:", outputs[0][0][intent_idx])
    return intent_labels[intent_idx]


C:\Users\Lenovo\AppData\Roaming\Python\Python39\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 🧠 Load Semantic Retriever (Keras)

In [2]:
from keras.saving import register_keras_serializable

@register_keras_serializable(package="Custom")
def triplet_loss(y_true, y_pred, margin=0.5):
    
    # Bagi menjadi tiga embedding sepanjang dim-1
    anchor, positive, negative = tf.split(y_pred, num_or_size_splits=3, axis=1)

    # Hitung squared Euclidean distance
    pos_dist = tf.reduce_sum(tf.square(anchor - positive), axis=1)
    neg_dist = tf.reduce_sum(tf.square(anchor - negative), axis=1)

    # Triplet loss
    basic_loss = pos_dist - neg_dist + margin
    loss = tf.reduce_mean(tf.maximum(basic_loss, 0.0))
    return loss

@register_keras_serializable(package="Custom")
def l2_normalize(t):
    return tf.math.l2_normalize(t, axis=1)

In [3]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import os
import pandas as pd

# Load model Siamese BiLSTM retriever
retriever_model = tf.keras.models.load_model(
    "../generated/siamese_model.keras",
    custom_objects={
        "triplet_loss": triplet_loss,
        "tf": tf,  # tambahkan ini
        "l2_normalize": l2_normalize
    }
)

# Load seluruh problem_description dari knowledge base CSV
folder_path = "../dataset/knowledge_base/"
all_data = []

for filename in os.listdir(folder_path):
    if filename.endswith(".csv"):
        df = pd.read_csv(os.path.join(folder_path, filename))
        all_data.append(df)

knowledge_base_df = pd.concat(all_data, ignore_index=True)

# Ambil semua teks symptom/problem_description
symptom_texts = knowledge_base_df["problem_description"].astype(str).tolist()

# Fit tokenizer dengan semua kalimat pada knowledge base
VOCAB_SIZE = 10000
OOV_TOKEN = "<OOV>"
retriver_tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
retriver_tokenizer.fit_on_texts(symptom_texts)

def embed_text(text):
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=30)
    return retriever_model.predict(pad)[0]

## 📚 Knowledge Base & Retrieval

In [4]:
encoder = retriever_model.get_layer("shared_encoder")
type(encoder)

keras.src.models.functional.Functional

In [5]:
import os
import pandas as pd
import numpy as np
import faiss
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Encoder dari retriever model
encoder = retriever_model.get_layer("shared_encoder")  # Sudah valid

# Tokenizer harus disiapkan sebelumnya: retriver_tokenizer

# Fungsi embedding teks
def embed_text(text):
    seq = retriver_tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=30)  # Sesuaikan maxlen dengan model retriever
    embedding = encoder.predict(pad)
    return embedding[0]

# Baca semua file CSV di folder knowledge_base
folder_path = "../dataset/knowledge_base/"
all_data = []

for filename in os.listdir(folder_path):
    if filename.endswith(".csv"):
        file_path = os.path.join(folder_path, filename)
        df = pd.read_csv(file_path)
        all_data.append(df)

# Gabungkan semua data
knowledge_base_df = pd.concat(all_data, ignore_index=True)

# Ubah menjadi list of dict
kbase = knowledge_base_df[["problem_description", "first_aid"]].rename(
    columns={"problem_description": "symptom"}
).to_dict(orient="records")

# Embedding semua data
kbase_vectors = np.array([embed_text(k["symptom"]) for k in kbase]).astype("float32")

# Bangun index FAISS
index = faiss.IndexFlatL2(kbase_vectors.shape[1])
index.add(kbase_vectors)

# Fungsi untuk retrieve solusi berdasarkan input user
RETRIEVAL_THRESHOLD = 0.4  # Atur eksperimen

def retrieve_solution(user_text):
    vec = embed_text(user_text).astype("float32").reshape(1, -1)
    D, I = index.search(vec, 1)
    distance = D[0][0]
    result = kbase[I[0][0]]
    
    print(f"[DEBUG] Jarak vektor: {distance}")
    
    if distance > RETRIEVAL_THRESHOLD:
        return {"symptom": None, "first_aid": "Mohon maaf, kami tidak menemukan solusi yang cukup relevan untuk masalah ini. Bisa dijelaskan lebih rinci?"}
    
    return result


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 983ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━

## 🔄 Full Chatbot Pipeline

In [6]:
def chatbot_reply(user_input):
    intent = predict_intent(user_input)

    if intent == "ask_first_aid_solution":
        result = retrieve_solution(user_input)
        return f"Pertolongan Pertama yang dapat Anda lakukan: {result['first_aid']}"
    
    elif intent == "ask_possible_cause":
        result = retrieve_solution(user_input)
        return f"Beberapa kemungkinan penyebabnya: {result}"

    elif intent == "report_noise_or_smell":
        result = retrieve_solution(user_input)
        return f"Terima kasih atas laporannya. Kami mendeteksi indikasi: {result}"

    elif intent == "casual_greeting":
        return "Halo! Ada keluhan atau masalah pada kendaraan Anda yang bisa saya bantu?"

    elif intent == "goodbye":
        return "Terima kasih! Semoga kendaraan Anda segera dalam kondisi baik."

    elif intent == "fallback":
        return "Maaf, saya belum memahami keluhan Anda. Bisa dijelaskan lebih lanjut?"

    else:
        return "Terjadi kesalahan dalam sistem. Silakan coba lagi atau hubungi teknisi kami."


## ✅ Tes Chatbot

In [10]:
# Tes input
user_input = "Kendaraan saya mengeluarkan suara aneh saat dinyalakan"
print("keluhan:", user_input, "->", chatbot_reply(user_input))

intent terdeteksi: report_noise_or_smell
confidence: 10.819839
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
[DEBUG] Jarak vektor: 0.37114253640174866
keluhan: Kendaraan saya mengeluarkan suara aneh saat dinyalakan -> Terima kasih atas laporannya. Kami mendeteksi indikasi: {'symptom': 'mobil nyendat saat akselerasi', 'first_aid': 'Coba periksa apakah sensor throttle bermasalah. Jika memungkinkan, bersihkan atau perbaiki sendiri menggunakan alat dasar. Pastikan kendaraan dalam keadaan mati dan aman sebelum mulai melakukan pengecekan. Misalnya, jika mobil nyendat saat akselerasi, cek sambungan kabel dan terminalnya.'}
